In [ ]:
import GalerkinToolkit as GT
import GLMakie as Makie
import DifferentialEquations as DE
import ProgressMeter as PM
using LinearAlgebra
using SparseArrays

In [ ]:
include(srcdir("FEM.jl"))
include(srcdir("equations.jl"))

In [1]:
import GLMakie as Makie
import GalerkinToolkit as GT
import Gmsh

In [2]:
"""
All equations used in the neural field model are defined in this file 
"""

using LinearAlgebra

function w(x,y)
    WAε(norm(x-y))
end

function f(u)
    μ=5.5
    θ=5.6
    1/(1+exp(-μ*u+θ)) - 1/(1+exp(θ))
end

function φ(r;)
    α=20
    β=1/α
    α / (cosh(β*norm(r)))^2
end
    
function A(x)
    b=0.4
    exp(-b*x)*(b*sin(x)+cos(x))
end

function WAε(x)
    ε=1.0e-3    
    Ax = A(x)
    abs(Ax) >= ε ? Ax : zero(Ax)
end

function τ(x,y) 
    #this function accepts node coordinates 
    τ_0 = 0.5
    τ_1 = 0.1
    diff = (x .- y).^2
    distance_xy = sqrt(sum(diff))
    τ_0 + τ_1*distance_xy
end 

function α(x,y, num_layer)
    #this function accepts node coordinates
    num_layer/τ(x,y)
end

α (generic function with 1 method)

In [3]:
function circle_mesh(gmsh, mesh_size, R)
    dim = 2
    gmsh.option.setNumber("General.Verbosity", 2)
    circle_tag = gmsh.model.occ.add_circle(0,0,0,R)
    circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
    circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
    gmsh.model.occ.synchronize()
    gmsh.model.model.add_physical_group(dim,[circle_surf_tag],-1,"cortex")
    gmsh.option.setNumber("Mesh.MeshSizeMax",mesh_size)
    gmsh.model.mesh.generate(dim)
    GT.mesh_from_gmsh(gmsh)
end

function plot_circle_mesh(Ω)
    """Plot circle mesh"""
    axis = (aspect = Makie.DataAspect(),)
    shading = Makie.NoShading
    fig = GT.makie_surfaces(Ω;color=:pink,axis,shading)
    GT.makie_edges!(Ω;color=:blue)
    Makie.save("domain.png", fig)
end 

plot_circle_mesh (generic function with 1 method)

In [4]:
### Create a mesh
mesh_size = 7
R = 30
axis = (aspect = Makie.DataAspect(),)
colormap=:viridis
mesh = GT.with_gmsh(gmsh -> circle_mesh(gmsh, mesh_size, R))
Ω = GT.interior(mesh)

### Plot a mesh - optional
plot_circle_mesh(Ω)

### Finite element interpolation
interpolation_degree = 1
V = GT.lagrange_space(Ω,interpolation_degree)
node_x = GT.node_coordinates(V)

89-element Vector{StaticArraysCore.SVector{2, Float64}}:
 [30.0, 0.0]
 [29.19134611739472, 6.918476122273196]
 [26.80897920970238, 13.463975406013844]
 [22.98133329356936, 19.28362829059616]
 [17.914757751083613, 24.06369578265129]
 [11.88239298117475, 27.5464832064082]
 [5.209445330007978, 29.54423259036623]
 [-1.7443448673141868, 29.949244748138053]
 [-8.604096981332598, 28.739685369464702]
 [-14.999999999999902, 25.980762113533213]
 ⋮
 [-9.681065650055624, -22.85043670950672]
 [-18.260132237507705, -15.50106845843651]
 [22.94544272362665, 8.586016466447685]
 [-11.724668210038892, -14.050443389695875]
 [-24.593735352289112, 5.568544436260571]
 [12.272994133952919, -22.016510854729475]
 [15.503508622949123, 9.737267944184268]
 [17.60275675933746, 18.75985494056673]
 [-9.915794502416363, 23.55159724729421]

In [12]:
function φ(r;)
    α=20
    β=1/α
    α / (cosh(β*norm(r)))^2
end

φ (generic function with 1 method)

In [13]:
initial_u = φ.(node_x)

89-element Vector{Float64}:
 3.6141327784729707
 3.6141327784729698
 3.6141327784729707
 3.6141327784729707
 3.6141327784729707
 3.6141327784729707
 3.6141327784729698
 3.6141327784729698
 3.6141327784729698
 3.6141327784729707
 ⋮
 5.696077761491793
 6.124665376589623
 5.850616664493987
 9.53041181134947
 5.5061648859629795
 5.510883098246122
 9.524943369109797
 5.271752940853969
 5.34975528360799

In [14]:
f(0)

0.0

In [11]:
α(node_x[4],node_x[89],1)

0.26196643770094846

In [ ]:
function initial_z(r)
    0
end

N = 5
t = ntuple(initial_z, N)
println(t)

In [ ]:
N = 3
len = 5

t = ntuple(i -> rand(len), N)   # Tuple{Vector{Float64}, ...} length N


In [ ]:
function z_initial(num_layer, dim_u)
    L = matrix(num_layer, dim_u)
    for i in dim_u
        L[i] = zeros()
    end    
end

In [ ]:
function z_initial(num_layer, dim_u)
    l_matrix = zeros(dim_u, num_layer)
    return l_matrix  
end 

In [ ]:
function z_initial(num_layer, dim_u)
    return [zeros(dim_u,dim_u) for _ in 1:num_layer] 
end 

function f(u)
    μ=5.5
    θ=5.6
    1/(1+exp(-μ*u+θ)) - 1/(1+exp(θ))
end

In [ ]:
matrix = z_initial(3,4)
F = [f.(m) for m in matrix]

In [ ]:
function τ(x,y)
    τ_0 = 0.5
    τ_1 = 1
    τ_0 + τ_1*abs(x-y)
end 

In [ ]:

function τ(x,y)
    τ_0 = 0.5
    τ_1 = 1
    diff = (x .- y).^2
    distance_xy = sqrt(sum(diff))
    τ_0 + τ_1*distance_xy
end

In [ ]:
τ([2,4], [-3,1])

In [ ]:
### Create a mesh
mesh_size = 7
R = 30
axis = (aspect = Makie.DataAspect(),)
colormap=:viridis
mesh = GT.with_gmsh(gmsh -> circle_mesh(gmsh, mesh_size, R))
Ω = GT.interior(mesh)
plot_circle_mesh(Ω)

### Finite element interpolation
interpolation_degree = 1
V = GT.lagrange_space(Ω,interpolation_degree)
node_x = GT.node_coordinates(V)